# Notebook 08 — Data Augmentation Deep Dive

This notebook is the **augmentation bible** for the rest of the course. For
every family of augmentations (spatial, colour, noise, policy-based, mix-based)
we show a *before/after* grid so you build a visual intuition for what each
transform does. We end with a mini experiment comparing weak vs strong
augmentation on a tiny Pet subset.

## Learning goals

- Understand **why** augmentation helps — the bias/variance trade-off.
- Know the five families: **Spatial, Colour, Noise, Policy-based, Mix-based**.
- Use `torchvision.transforms.v2` and `albumentations` fluently.
- Implement **Mixup** and **CutMix** and their losses from scratch.
- Use **label smoothing** and know why it is a "soft-label" regulariser.
- Run a quick experiment that actually *measures* the benefit of strong aug.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/08_augmentation_deep_dive.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Why augment?

Training a deep network on a fixed dataset is a **bias / variance** balancing
act:

- A tiny model has **high bias** — it cannot represent the data well.
- A giant model has **low bias** but **high variance** — it will memorise the
  1,000 training images (overfit) instead of learning generalisable features.

Augmentation attacks the variance side. If you have 1,000 training images and
apply random flips, crops, colour jitter, and RandAugment, the effective
number of distinct images the model sees across 100 epochs is closer to
**~100,000**. The model can no longer memorise individual pixels — it is
forced to learn invariant, higher-level features.

Augmentation is therefore **regularisation via data diversity**.

## 2. The augmentation zoo

| Family | What it changes | Examples |
|--------|-----------------|----------|
| **Spatial** | geometry / pixel positions | Flip, Rotation, RandomResizedCrop, Affine, Perspective |
| **Colour**  | per-pixel colour values | ColorJitter, Grayscale, Invert, Posterize |
| **Noise / occlusion** | add corruption or cover regions | RandomErasing, GaussianBlur, CoarseDropout |
| **Policy-based** | pick & chain ops from a learned/random policy | AutoAugment, RandAugment, TrivialAugmentWide |
| **Mix-based**   | combine **two** images into one training sample | Mixup, CutMix |

All of them are compatible with `torchvision.transforms.v2`, and the
noise/occlusion and many spatial ops are faster or richer in `albumentations`.

## 3. Helpers — load one image and repeatedly apply a transform

To visualise randomness, we call each transform **8 times** on the same image
and look at all 8 outputs side by side.

In [ ]:
import os
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import functional as TF
from utils.env import data_dir
from utils.plotting import show_image_grid

def _find_one_image() -> Image.Image:
    """Return one RGB image from the Pet dataset if available, else synthetic."""
    root = data_dir()
    for folder in ("oxford-iiit-pet", "oxford_iiit_pet", "pets"):
        p = os.path.join(root, folder)
        if os.path.isdir(p):
            for r, _, files in os.walk(p):
                for f in sorted(files):
                    if f.lower().endswith((".jpg", ".jpeg", ".png")):
                        return Image.open(os.path.join(r, f)).convert("RGB")
    # synthetic fallback: coloured diagonal gradient
    a = np.zeros((256, 256, 3), dtype=np.uint8)
    yy, xx = np.mgrid[0:256, 0:256]
    a[..., 0] = xx; a[..., 1] = yy; a[..., 2] = (255 - (xx + yy) // 2).clip(0, 255)
    return Image.fromarray(a)

pil_img = _find_one_image().resize((224, 224))
print("sample image size:", pil_img.size)

def apply_many(transform, pil_image, n: int = 8):
    """Apply a torchvision transform n times; return a list of (C, H, W) tensors in [0, 1]."""
    out = []
    for _ in range(n):
        img = transform(pil_image)
        if isinstance(img, Image.Image):
            img = TF.to_tensor(img)
        elif isinstance(img, torch.Tensor):
            if img.dtype != torch.float32:
                img = img.float() / 255.0
            if img.max() > 1.5:
                img = img / 255.0
        out.append(img.clamp(0, 1).cpu())
    return out

# sanity — show the original
show_image_grid([TF.to_tensor(pil_img)], titles=["original"], ncols=1)

## 4. Spatial augmentations

Spatial ops move pixels around **without** changing their colour values.
We use `torchvision.transforms.v2` — the new API that is ~2x faster than v1
for tensor inputs and supports bounding boxes and masks.

In [ ]:
from torchvision.transforms import v2 as T

spatial_transforms = {
    "HorizontalFlip":     T.RandomHorizontalFlip(p=0.8),
    "VerticalFlip":       T.RandomVerticalFlip(p=0.8),
    "Rotation(20)":       T.RandomRotation(degrees=20),
    "RandomResizedCrop":  T.RandomResizedCrop(size=224, scale=(0.5, 1.0)),
    "Affine":             T.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    "Perspective":        T.RandomPerspective(distortion_scale=0.5, p=1.0),
}

for name, tr in spatial_transforms.items():
    print(f"\n### {name}")
    imgs = apply_many(tr, pil_img, n=8)
    show_image_grid(imgs, titles=[f"{name} #{i}" for i in range(8)], ncols=4)

**Tips.**

- `RandomResizedCrop` is by far the most important single spatial aug for
  ImageNet-style training. It teaches scale invariance.
- `VerticalFlip` is **rarely** used for natural images (cats don't fall from
  the sky) but is handy for medical / satellite / texture data.
- `RandomRotation` with a large angle can push class-defining features out of
  the crop; keep it modest (15-30 degrees) for photos.

## 5. Colour augmentations

Colour ops change per-pixel values without moving pixels. They teach the
model to be invariant to lighting, white-balance, and camera-specific
colour response.

In [ ]:
colour_transforms = {
    "ColorJitter":      T.ColorJitter(brightness=0.5, contrast=0.5,
                                       saturation=0.5, hue=0.1),
    "RandomGrayscale":  T.RandomGrayscale(p=0.5),
    "RandomInvert":     T.RandomInvert(p=0.5),
    "RandomPosterize":  T.RandomPosterize(bits=3, p=1.0),
}

for name, tr in colour_transforms.items():
    print(f"\n### {name}")
    imgs = apply_many(tr, pil_img, n=8)
    show_image_grid(imgs, titles=[f"{name} #{i}" for i in range(8)], ncols=4)

**Tips.**

- `ColorJitter` is the most frequently used colour aug. Start with values in
  `0.1–0.3` and grow from there.
- `RandomPosterize` keeps only the top-k bits per channel — a handy trick
  for adversarial robustness research.
- `RandomInvert` is useful for datasets where polarity is meaningless
  (icons, line art), but can destroy semantics for natural photos.

## 6. Noise & occlusion

These augmentations add corruption or hide regions. They force the model to
rely on **multiple** cues rather than a single discriminative patch.

In [ ]:
# torchvision.RandomErasing operates on tensors, so we wrap it accordingly.
from torchvision.transforms import v2 as T

class TensorPipeline:
    """Helper: PIL -> Tensor -> transform -> Tensor."""
    def __init__(self, tensor_tr):
        self.tensor_tr = tensor_tr
    def __call__(self, pil_image):
        t = TF.to_tensor(pil_image)
        return self.tensor_tr(t)

random_erasing = T.Compose([
    T.RandomErasing(p=1.0, scale=(0.05, 0.25), ratio=(0.3, 3.3), value=0.0),
])

gaussian_blur = T.GaussianBlur(kernel_size=11, sigma=(0.5, 4.0))

noise_transforms = {
    "RandomErasing": TensorPipeline(random_erasing),
    "GaussianBlur":  gaussian_blur,
}

for name, tr in noise_transforms.items():
    print(f"\n### {name}")
    imgs = apply_many(tr, pil_img, n=8)
    show_image_grid(imgs, titles=[f"{name} #{i}" for i in range(8)], ncols=4)

In [ ]:
# Albumentations — CoarseDropout (random rectangular holes). Handy because
# it works natively with numpy uint8 and also handles bboxes / masks.
try:
    import albumentations as A
    import numpy as np

    aug = A.Compose([
        A.CoarseDropout(
            max_holes=8, max_height=32, max_width=32,
            min_holes=4, min_height=16, min_width=16,
            fill_value=0, p=1.0,
        ),
    ])

    def albu_apply(pil_image):
        arr = np.array(pil_image)
        out = aug(image=arr)["image"]
        return torch.from_numpy(out).permute(2, 0, 1).float() / 255.0

    imgs = [albu_apply(pil_img) for _ in range(8)]
    print("### albumentations.CoarseDropout")
    show_image_grid(imgs, titles=[f"hole #{i}" for i in range(8)], ncols=4)
except ImportError:
    print("albumentations not installed — skipping CoarseDropout demo.")
    print("  pip install albumentations")

## 7. Policy-based augmentations — the SOTA family

Writing a good augmentation pipeline by hand is tedious. The **policy** family
automates it:

- **AutoAugment** (2019) — a policy (list of `(op, magnitude, prob)` triples)
  **learned** via RL on a proxy task. Each call picks a random sub-policy.
- **RandAugment** (2020) — forget the learning; just pick `N` ops uniformly at
  random from a fixed list, each applied at a single magnitude `M`. Two hyper
  parameters, two numbers on a slider.
- **TrivialAugmentWide** (2021) — even simpler: pick **one** op, random magnitude.
  Often matches or beats RandAugment with zero tuning.

These are typically the single biggest accuracy win you can get from changing
augmentation.

In [ ]:
from torchvision.transforms import v2 as T

policy_transforms = {
    "AutoAugment(ImageNet)":   T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET),
    "RandAugment(N=2,M=9)":    T.RandAugment(num_ops=2, magnitude=9),
    "TrivialAugmentWide":      T.TrivialAugmentWide(),
}

for name, tr in policy_transforms.items():
    print(f"\n### {name}")
    imgs = apply_many(tr, pil_img, n=8)
    show_image_grid(imgs, titles=[f"{name} #{i}" for i in range(8)], ncols=4)

## 8. Mix-based: Mixup

**Mixup** (2017) creates a new training sample by linearly interpolating
**two** images *and* their labels:

```text
  lam ~ Beta(alpha, alpha)           # typically alpha = 0.2
  x_mix = lam * x_A + (1 - lam) * x_B
  loss  = lam * CE(pred, y_A) + (1 - lam) * CE(pred, y_B)
```

Intuition: we forbid the network from being 100% confident on any one-hot
label, so it learns to produce calibrated, smoother decision boundaries.

In [ ]:
# Demo: grab a second PIL image so we can mix two real pictures.
pil_a = pil_img
# Build a visibly different second image by colour-inverting the first.
pil_b = Image.fromarray(255 - np.array(pil_a))

x_A = TF.to_tensor(pil_a)   # (3, 224, 224)
x_B = TF.to_tensor(pil_b)

def mixup_pair(x_a, x_b, alpha: float = 0.2):
    lam = float(np.random.beta(alpha, alpha))
    x_mix = lam * x_a + (1.0 - lam) * x_b
    return x_mix, lam

mixed = []
titles = []
for _ in range(8):
    x_mix, lam = mixup_pair(x_A, x_B, alpha=0.2)
    mixed.append(x_mix)
    titles.append(f"lam={lam:.2f}")

show_image_grid(mixed, titles=titles, ncols=4)

In [ ]:
# The corresponding loss. Given a minibatch, we shuffle it to get (A, B) pairs.
import torch.nn.functional as F

def mixup_batch(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.2):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1.0 - lam) * x[perm]
    y_a, y_b = y, y[perm]
    return x_mix, y_a, y_b, lam

def mixup_loss(logits: torch.Tensor, y_a, y_b, lam: float) -> torch.Tensor:
    return lam * F.cross_entropy(logits, y_a) + (1.0 - lam) * F.cross_entropy(logits, y_b)

# Smoke-test: fake minibatch of 4 samples, 10 classes.
fake_x      = torch.randn(4, 3, 32, 32)
fake_y      = torch.tensor([0, 3, 7, 1])
fake_logits = torch.randn(4, 10)

x_mix, y_a, y_b, lam = mixup_batch(fake_x, fake_y, alpha=0.2)
print(f"lam = {lam:.3f}")
print("y_a:", y_a.tolist(), "y_b:", y_b.tolist())
print("mixup loss:", mixup_loss(fake_logits, y_a, y_b, lam).item())

## 9. Mix-based: CutMix

**CutMix** (2019) cuts a random rectangle out of image B and pastes it into
image A. The label weight is the *area ratio* of the pasted region:

```text
  lam_area = H_box * W_box / (H * W)
  x_mix    = A, with the box region replaced by B
  loss     = (1 - lam_area) * CE(pred, y_A) + lam_area * CE(pred, y_B)
```

Why it often beats Mixup for classification: the model sees **non-blended**
pixels, so it still learns sharp local features — but forced attention is
distributed over a larger region of the image.

In [ ]:
def rand_bbox(H: int, W: int, lam: float):
    """Return a random box with area (1 - lam) * H * W."""
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cy = np.random.randint(H)
    cx = np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2

def cutmix_pair(x_a: torch.Tensor, x_b: torch.Tensor, alpha: float = 1.0):
    lam = float(np.random.beta(alpha, alpha))
    _, H, W = x_a.shape
    y1, y2, x1, x2 = rand_bbox(H, W, lam)
    x_mix = x_a.clone()
    x_mix[:, y1:y2, x1:x2] = x_b[:, y1:y2, x1:x2]
    # adjust lam to match the *actual* area (boxes clipped at borders)
    lam_area = 1.0 - ((y2 - y1) * (x2 - x1) / (H * W))
    return x_mix, lam_area

mixed, titles = [], []
for _ in range(8):
    x_mix, lam_area = cutmix_pair(x_A, x_B, alpha=1.0)
    mixed.append(x_mix)
    titles.append(f"lam_area={lam_area:.2f}")
show_image_grid(mixed, titles=titles, ncols=4)

In [ ]:
def cutmix_batch(x, y, alpha: float = 1.0):
    lam = float(np.random.beta(alpha, alpha))
    B, _, H, W = x.shape
    perm = torch.randperm(B, device=x.device)
    y1, y2, x1, x2 = rand_bbox(H, W, lam)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam_area = 1.0 - ((y2 - y1) * (x2 - x1) / (H * W))
    y_a, y_b = y, y[perm]
    return x_mix, y_a, y_b, lam_area

def cutmix_loss(logits, y_a, y_b, lam_area):
    return lam_area * F.cross_entropy(logits, y_a) + (1.0 - lam_area) * F.cross_entropy(logits, y_b)

x_mix, y_a, y_b, lam_area = cutmix_batch(fake_x, fake_y, alpha=1.0)
print(f"lam_area = {lam_area:.3f}")
print("cutmix loss:", cutmix_loss(fake_logits, y_a, y_b, lam_area).item())

## 10. Label smoothing — soft labels for free

One-hot targets encourage the model to push the winning logit to `+infty`,
which leads to over-confident, poorly calibrated predictions. **Label
smoothing** replaces the hard target with a distribution that gives `1 - eps`
to the correct class and spreads `eps` uniformly across the others
(usually `eps = 0.1`).

PyTorch bakes it into `CrossEntropyLoss` via the `label_smoothing=` argument —
no extra code required.

In [ ]:
# Tiny demonstration: how the loss value changes at the same logits.
logits = torch.tensor([[5.0, 1.0, -1.0]])   # model very confident on class 0
target = torch.tensor([0])

loss_hard = F.cross_entropy(logits, target, label_smoothing=0.0)
loss_soft = F.cross_entropy(logits, target, label_smoothing=0.1)
print(f"hard-label CE    : {loss_hard.item():.4f}")
print(f"smoothed CE (0.1): {loss_soft.item():.4f}")

# With smoothing, the loss is larger even when the model picks the right class —
# it wants the model to leave some probability mass for wrong classes.
print("smoothing penalises over-confidence -> better calibration.")

## 11. `torchvision.transforms.v2` vs `albumentations`

Both libraries cover the same *concept* space but their engineering trade-offs
differ.

| Aspect | torchvision v2 | albumentations |
|--------|----------------|----------------|
| Native tensor input | fast (GPU-ready) | cpu / numpy only |
| Bbox / mask aware | yes (v2!) | yes (original author's strength) |
| Library size | comes with torchvision | `pip install albumentations` |
| API style | functional OOP | imperative, `A.Compose` |
| Unique ops | none particularly unique | `RandomGamma`, `HueSaturationValue`, many more weather/noise ops |

**Rule of thumb.** If your pipeline stays on tensors, use v2. If you need a
richer catalogue of noise/weather/medical augs or multi-modal (image + mask +
bbox + keypoints) transforms, use albumentations. Both compose with PyTorch
datasets the same way.

In [ ]:
# Two equivalent pipelines — same semantic content, different libraries.

# --- torchvision.transforms.v2 ---
tv_pipeline = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.PILToTensor(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])
print("torchvision v2 pipeline:")
print(tv_pipeline)

# --- albumentations ---
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    al_pipeline = A.Compose([
        A.RandomResizedCrop(height=224, width=224),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std =(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    print("\nalbumentations pipeline:")
    print(al_pipeline)
except ImportError:
    print("albumentations not installed — skipping.")

## 12. Mini experiment — weak vs strong augmentation

To turn intuition into numbers, we train the **same** tiny CNN twice on a
small Pet subset (500 train / 100 val) for 5 epochs and compare validation
accuracy:

- **weak**: Resize + ToTensor + Normalize.
- **strong**: weak + RandAugment + RandomErasing + label_smoothing=0.1.

With only 500 training images the risk of overfitting is extreme, so we
expect strong augmentation to help. If you are running locally without
the Pet dataset, the cells below transparently fall back to a synthetic
random-noise dataset so the code still executes end-to-end.

In [ ]:
# ----- Build the datasets (Pet if available, else synthetic fallback) -----
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms as Tv1
from utils.env import data_dir

IMG_SIZE = 96   # small to keep the experiment fast on CPU
NUM_CLASSES = 37  # Oxford Pet has 37 classes; synthetic fallback uses 5

WEAK_TFM = Tv1.Compose([
    Tv1.Resize((IMG_SIZE, IMG_SIZE)),
    Tv1.ToTensor(),
    Tv1.Normalize(mean=[0.485, 0.456, 0.406],
                  std =[0.229, 0.224, 0.225]),
])

STRONG_TFM = Tv1.Compose([
    Tv1.Resize((IMG_SIZE, IMG_SIZE)),
    Tv1.RandAugment(num_ops=2, magnitude=9),
    Tv1.ToTensor(),
    Tv1.Normalize(mean=[0.485, 0.456, 0.406],
                  std =[0.229, 0.224, 0.225]),
    Tv1.RandomErasing(p=0.5, scale=(0.05, 0.25)),
])

# --- Try Oxford Pet ---
def build_pet_datasets():
    from torchvision.datasets import OxfordIIITPet
    root = data_dir()
    try:
        full = OxfordIIITPet(root=root, split="trainval", download=False)
    except Exception as e:
        print("Pet dataset not found (", e, ") - will fall back to synthetic.")
        return None
    # take 500 train + 100 val from the first 600 samples
    idx_train = list(range(500))
    idx_val   = list(range(500, 600))
    # Two copies with different transforms.
    class PetTransformed(Dataset):
        def __init__(self, base, indices, tfm):
            self.base, self.indices, self.tfm = base, indices, tfm
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            img, y = self.base[self.indices[i]]
            return self.tfm(img), y
    return (
        PetTransformed(full, idx_train, WEAK_TFM),
        PetTransformed(full, idx_train, STRONG_TFM),
        PetTransformed(full, idx_val, WEAK_TFM),
        37,
    )

# --- Synthetic fallback ---
class SyntheticDS(Dataset):
    """Each class is a solid colour tile with per-sample noise. 5 classes."""
    def __init__(self, n: int, tfm, num_classes: int = 5, seed: int = 0):
        self.n, self.tfm, self.k = n, tfm, num_classes
        rng = np.random.default_rng(seed)
        self.labels = rng.integers(0, num_classes, size=n).tolist()
        # base colour per class
        self.base_colours = (rng.random((num_classes, 3)) * 255).astype(np.uint8)
    def __len__(self): return self.n
    def __getitem__(self, i):
        lbl = self.labels[i]
        arr = np.tile(self.base_colours[lbl], (128, 128, 1)).astype(np.int16)
        arr += np.random.randint(-30, 30, arr.shape, dtype=np.int16)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
        return self.tfm(img), lbl

def build_synthetic_datasets():
    return (
        SyntheticDS(500, WEAK_TFM,   num_classes=5, seed=1),
        SyntheticDS(500, STRONG_TFM, num_classes=5, seed=1),
        SyntheticDS(100, WEAK_TFM,   num_classes=5, seed=2),
        5,
    )

bundle = build_pet_datasets()
if bundle is None:
    print("Using synthetic dataset.")
    train_weak, train_strong, val_ds, NUM_CLASSES = build_synthetic_datasets()
else:
    print("Using Oxford Pet subset.")
    train_weak, train_strong, val_ds, NUM_CLASSES = bundle

print(f"train: {len(train_weak)}  val: {len(val_ds)}  classes: {NUM_CLASSES}")

In [ ]:
# ----- A tiny CNN (same architecture in both runs, different seeds via rng) -----
class TinyCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                nn.BatchNorm2d(c_out),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32),   # 96 -> 48
            block(32, 64),  # 48 -> 24
            block(64, 128), # 24 -> 12
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

# smoke test
m = TinyCNN(NUM_CLASSES)
print(m)
print("forward ok:", m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape)

In [ ]:
# ----- Training helpers -----
def run_experiment(train_ds, val_ds, num_classes: int,
                   epochs: int = 5, lr: float = 3e-3,
                   label_smoothing: float = 0.0,
                   batch_size: int = 32, seed: int = 0):
    torch.manual_seed(seed)
    model = TinyCNN(num_classes).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=0)

    history = {"train_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), torch.as_tensor(yb).to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * xb.size(0); n += xb.size(0)
        tl = loss_sum / max(n, 1)

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), torch.as_tensor(yb).to(device)
                pred = model(xb).argmax(1)
                correct += (pred == yb).sum().item()
                total   += yb.size(0)
        va = correct / max(total, 1)
        history["train_loss"].append(tl)
        history["val_acc"].append(va)
        print(f"epoch {epoch}: train_loss={tl:.4f}  val_acc={va:.4f}")
    return history

In [ ]:
print("=== WEAK augmentation ===")
hist_weak = run_experiment(train_weak, val_ds, NUM_CLASSES,
                           epochs=5, label_smoothing=0.0, seed=0)

print("\n=== STRONG augmentation (+ label smoothing 0.1) ===")
hist_strong = run_experiment(train_strong, val_ds, NUM_CLASSES,
                             epochs=5, label_smoothing=0.1, seed=0)

In [ ]:
# ----- Plot overlaid validation-accuracy curves -----
import matplotlib.pyplot as plt

epochs_axis = list(range(1, len(hist_weak["val_acc"]) + 1))
plt.figure(figsize=(6, 4))
plt.plot(epochs_axis, hist_weak["val_acc"],   marker="o", label="weak aug")
plt.plot(epochs_axis, hist_strong["val_acc"], marker="s", label="strong aug + LS")
plt.xlabel("epoch"); plt.ylabel("val accuracy")
plt.title("Weak vs strong augmentation on a 500-image subset")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

**How to read the result.** On small datasets strong augmentation usually
wins by 3–10 accuracy points. If you see the two lines crossing or touching,
it is a useful reminder that augmentation is **not free** — aggressive aug on
an *already-big* dataset can slightly hurt short-horizon training. Rule of
thumb: the smaller your dataset, the harder you should augment.

## Key Takeaways

- Augmentation is **regularisation via data diversity** — the smaller your
  dataset, the more you should augment.
- Five families cover >95% of what you need: **Spatial, Colour, Noise,
  Policy-based, Mix-based**.
- `RandomResizedCrop` + horizontal flip is the **minimum viable** pipeline
  for natural images.
- Policy-based augs (`AutoAugment`, `RandAugment`, `TrivialAugmentWide`) are
  the biggest single accuracy win you can get for free.
- **Mixup** and **CutMix** sample two images and interpolate their labels.
  Pair them with **label smoothing** for well-calibrated outputs.
- `torchvision.transforms.v2` is fast and native to PyTorch.
  `albumentations` is richer and multi-modal. Pick either — they compose the
  same way with Datasets.
- A tiny experiment on 500 images typically shows that strong aug > weak aug
  in validation accuracy.

## Exercises

1. **Compose your own policy.** Using `T.RandomApply([...], p=...)` build an
   aug pipeline that applies ColorJitter and GaussianBlur each 30% of the
   time, plus RandAugment always. Visualise 16 samples.
2. **Inspect the Beta distribution.** For `alpha in {0.1, 0.2, 1.0, 2.0}`
   draw 10,000 samples from `Beta(alpha, alpha)` and plot histograms.
   For what values is Mixup **mostly** close to un-mixed images?
3. **Bbox-aware augmentation.** Using `albumentations` write a pipeline that
   takes an image and a bounding box; flip, crop, and jitter the image while
   keeping the bbox coordinates consistent.
4. **Longer experiment.** Run the mini experiment in Section 12 for 20
   epochs, three seeds each, and plot mean accuracy with shaded min/max.
   Does strong aug still win?
5. **Custom MixUp+CutMix.** Write a single collate_fn that randomly picks
   Mixup or CutMix per minibatch (50/50) and returns `(x_mix, y_a, y_b, lam)`.
   Plug it into `DataLoader(collate_fn=...)`.